In [ ]:
!pip install -q requests beautifulsoup4 pandas



In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import sqlite3
import re
from pathlib import Path

BASE_URL = "https://books.toscrape.com"
FIXED_RATE = 105.50

print("Setup complete")
print("Fixed rate: 1 GBP =", FIXED_RATE, "INR")

Setup complete
Fixed rate: 1 GBP = 105.5 INR


In [ ]:
CATEGORY_URLS = {
    "Mystery": f"{BASE_URL}/catalogue/category/books/mystery_3/index.html",
    "Young Fiction": f"{BASE_URL}/catalogue/category/books/young-adult_21/index.html",
    "Default": f"{BASE_URL}/catalogue/category/books/default_15/index.html"
}

CATEGORY_URLS

{'Mystery': 'https://books.toscrape.com/catalogue/category/books/mystery_3/index.html',
 'Young Fiction': 'https://books.toscrape.com/catalogue/category/books/young-adult_21/index.html',
 'Default': 'https://books.toscrape.com/catalogue/category/books/default_15/index.html'}

In [ ]:
def scrape_category(category, url):
    response = requests.get(
        url,
        timeout=20,
        headers={"User-Agent": "Mozilla/5.0"}
    )

    print(category, "Status Code:", response.status_code)

    response.raise_for_status()

    soup = BeautifulSoup(response.text, "html.parser")

    books = []

    for article in soup.select("article.product_pod"):

        title_tag = article.select_one("h3 a")
        price_tag = article.select_one(".price_color")
        rating_tag = article.select_one("p.star-rating")
        availability_tag = article.select_one(".availability")

        books.append({
            "title": title_tag.get("title"),
            "price_raw": price_tag.get_text(strip=True),
            "star_rating": " ".join(rating_tag.get("class", [])),
            "availability": availability_tag.get_text(" ", strip=True),
            "category": category
        })

    return books

In [ ]:
raw_books = []

for category, url in CATEGORY_URLS.items():
    category_books = scrape_category(category, url)
    raw_books.extend(category_books)

print("Total books scraped:", len(raw_books))

Mystery Status Code: 200
Young Fiction Status Code: 200
Default Status Code: 200
Total books scraped: 60


In [ ]:
df = pd.DataFrame(raw_books)

df.head()

,title,price_raw,star_rating,availability,category
0,Sharp Objects,Â£47.82,star-rating Four,In stock,Mystery
1,"In a Dark, Dark Wood",Â£19.63,star-rating One,In stock,Mystery
2,The Past Never Ends,Â£56.50,star-rating Four,In stock,Mystery
3,A Murder in Time,Â£16.64,star-rating One,In stock,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),Â£44.10,star-rating Four,In stock,Mystery


In [ ]:
def parse_price(value):
    try:
        match = re.search(r"£\s*([0-9]+(?:\.[0-9]+)?)", str(value))

        if match:
            return float(match.group(1))

        return None

    except Exception:
        return None


df["price_gbp"] = df["price_raw"].apply(parse_price)

df[["price_raw", "price_gbp"]].head()

,price_raw,price_gbp
0,Â£47.82,47.82
1,Â£19.63,19.63
2,Â£56.50,56.50
3,Â£16.64,16.64
4,Â£44.10,44.10


In [ ]:
rating_map = {
    "One": 1,
    "Two": 2,
    "Three": 3,
    "Four": 4,
    "Five": 5
}

def parse_rating(value):
    try:
        rating_name = str(value).split()[-1]
        return rating_map.get(rating_name)
    except Exception:
        return None


df["rating"] = df["star_rating"].apply(parse_rating)

df[["star_rating", "rating"]].head()

,star_rating,rating
0,star-rating Four,4
1,star-rating One,1
2,star-rating Four,4
3,star-rating One,1
4,star-rating Four,4


In [ ]:
def parse_stock(value):
    value = str(value).strip().lower()

    if value.startswith("in stock"):
        return True

    if value.startswith("out of stock"):
        return False

    return None


df["in_stock"] = df["availability"].apply(parse_stock)

df[["availability", "in_stock"]].head()

,availability,in_stock
0,In stock,True
1,In stock,True
2,In stock,True
3,In stock,True
4,In stock,True


In [ ]:
# Numeric fields: median imputation
df["price_gbp"] = df["price_gbp"].fillna(df["price_gbp"].median())
df["rating"] = df["rating"].fillna(df["rating"].median())

# Rating must be integer 1-5
df["rating"] = (
    df["rating"]
    .round()
    .clip(1, 5)
    .astype(int)
)

# Availability is boolean/non-numeric.
# If it cannot be parsed, drop that row.
df = df.dropna(subset=["in_stock"]).copy()

df["in_stock"] = df["in_stock"].astype(bool)

print("Rows after cleaning:", len(df))

Rows after cleaning: 60


In [ ]:
df["price_inr"] = (df["price_gbp"] * FIXED_RATE).round(2)

df = df[
    [
        "title",
        "price_gbp",
        "price_inr",
        "rating",
        "in_stock",
        "category"
    ]
]

df.head(10)

,title,price_gbp,price_inr,rating,in_stock,category
0,Sharp Objects,47.82,5045.01,4,True,Mystery
1,"In a Dark, Dark Wood",19.63,2070.96,1,True,Mystery
2,The Past Never Ends,56.50,5960.75,4,True,Mystery
3,A Murder in Time,16.64,1755.52,1,True,Mystery
4,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.55,4,True,Mystery
5,The Last Mile (Amos Decker #2),54.21,5719.16,2,True,Mystery
6,That Darkness (Gardiner and Renner #1),13.92,1468.56,1,True,Mystery
7,Tastes Like Fear (DI Marnie Rome #3),10.69,1127.79,1,True,Mystery
8,A Time of Torment (Charlie Parker #14),48.35,5100.92,5,True,Mystery
9,A Study in Scarlet (Sherlock Holmes #1),16.73,1765.02,2,True,Mystery


In [ ]:
print("Number of books:", len(df))
print("Number of categories:", df["category"].nunique())
print("\nCategories:")
print(df["category"].unique())

print("\nData types:")
print(df.dtypes)

Number of books: 60
Number of categories: 3

Categories:
['Mystery' 'Young Fiction' 'Default']

Data types:
title         object
price_gbp    float64
price_inr    float64
rating         int64
in_stock        bool
category      object
dtype: object


In [ ]:
import sqlite3

# Create SQLite database
conn = sqlite3.connect("books_catalog.db")

# Enable foreign keys
conn.execute("PRAGMA foreign_keys = ON")

# Create normalized tables
conn.executescript("""
DROP TABLE IF EXISTS books;
DROP TABLE IF EXISTS categories;

CREATE TABLE categories (
    category_id INTEGER PRIMARY KEY,
    category_name TEXT NOT NULL UNIQUE
);

CREATE TABLE books (
    book_id INTEGER PRIMARY KEY,
    title TEXT NOT NULL,
    price_gbp REAL NOT NULL,
    price_inr REAL NOT NULL,
    rating INTEGER NOT NULL CHECK (rating BETWEEN 1 AND 5),
    in_stock INTEGER NOT NULL CHECK (in_stock IN (0, 1)),
    category_id INTEGER NOT NULL,
    FOREIGN KEY (category_id)
        REFERENCES categories(category_id)
);
""")

print("SQLite tables created successfully!")

SQLite tables created successfully!


In [ ]:
# Insert unique categories
categories = df["category"].unique()

for category in categories:
    conn.execute(
        "INSERT INTO categories (category_name) VALUES (?)",
        (category,)
    )

conn.commit()

# Check categories
pd.read_sql(
    "SELECT * FROM categories",
    conn
)

,category_id,category_name
0,1,Mystery
1,2,Young Fiction
2,3,Default


In [ ]:
# Get category IDs
category_map = pd.read_sql(
    "SELECT category_id, category_name FROM categories",
    conn
)

category_map

# Create mapping: category name -> category ID
category_dict = dict(
    zip(
        category_map["category_name"],
        category_map["category_id"]
    )
)

# Insert books
for _, row in df.iterrows():

    conn.execute(
        """
        INSERT INTO books
        (
            title,
            price_gbp,
            price_inr,
            rating,
            in_stock,
            category_id
        )
        VALUES (?, ?, ?, ?, ?, ?)
        """,
        (
            row["title"],
            row["price_gbp"],
            row["price_inr"],
            row["rating"],
            int(row["in_stock"]),
            category_dict[row["category"]]
        )
    )

conn.commit()

print("Books inserted successfully!")

Books inserted successfully!


In [ ]:
print("Categories:")
display(pd.read_sql("SELECT * FROM categories", conn))

print("\nNumber of books:")
print(pd.read_sql("SELECT COUNT(*) AS total_books FROM books", conn))

Categories:


,category_id,category_name
0,1,Mystery
1,2,Young Fiction
2,3,Default



Number of books:
   total_books
0           60


In [ ]:
query1 = """
SELECT title, price_gbp, rating, in_stock
FROM books
WHERE in_stock = 1
  AND price_gbp BETWEEN 10 AND 20
ORDER BY price_gbp ASC;
"""

result1 = pd.read_sql(query1, conn)

display(result1)

,title,price_gbp,rating,in_stock
0,Tastes Like Fear (DI Marnie Rome #3),10.69,1,1
1,Hide Away (Eve Duncan #20),11.84,1,1
2,Playing with Fire,13.71,3,1
3,That Darkness (Gardiner and Renner #1),13.92,1,1
4,"Starving Hearts (Triangular Trade Trilogy, #1)",13.99,2,1
5,Wild Swans,14.36,2,1
6,The Epidemic (The Program 0.6),14.44,5,1
7,Obsidian (Lux #1),14.86,2,1
8,A Murder in Time,16.64,1,1
9,A Study in Scarlet (Sherlock Holmes #1),16.73,2,1


In [ ]:
query2 = """
SELECT title, price_gbp, rating
FROM books
ORDER BY price_gbp DESC
LIMIT 10;
"""

result2 = pd.read_sql(query2, conn)

display(result2)

,title,price_gbp,rating
0,Boar Island (Anna Pigeon #19),59.48,3
1,Aristotle and Dante Discover the Secrets of th...,58.14,4
2,"A Piece of Sky, a Grain of Rice: A Memoir in F...",56.76,5
3,The Past Never Ends,56.50,4
4,Don't Get Caught,55.35,1
5,The Girl on the Train,55.02,2
6,Murder at the 42nd Street Library (Raymond Amb...,54.36,4
7,The Last Mile (Amos Decker #2),54.21,2
8,Aladdin and His Wonderful Lamp,53.13,3
9,Thirteen Reasons Why,52.72,1


In [ ]:
query3 = """
SELECT DISTINCT c.category_name
FROM categories AS c
JOIN books AS b
    ON c.category_id = b.category_id
ORDER BY c.category_name;
"""

result3 = pd.read_sql(query3, conn)

display(result3)

,category_name
0,Default
1,Mystery
2,Young Fiction


In [ ]:
query4 = """
SELECT title, rating, price_gbp
FROM books
WHERE rating IN (4, 5)
ORDER BY rating DESC, price_gbp DESC;
"""

result4 = pd.read_sql(query4, conn)

display(result4)

,title,rating,price_gbp
0,"A Piece of Sky, a Grain of Rice: A Memoir in F...",5,56.76
1,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30
2,Library of Souls (Miss Peregrineâs Peculiar ...,5,48.56
3,A Time of Torment (Charlie Parker #14),5,48.35
4,Scarlett Epstein Hates It Here,5,43.55
5,The Darkest Lie,5,35.35
6,Frostbite (Vampire Academy #2),5,29.99
7,What Happened on Beale Street (Secrets of the ...,5,25.37
8,The Inefficiency Assassin: Time Management Tac...,5,20.59
9,Set Me Free,5,17.46


In [ ]:
query5 = """
SELECT
    c.category_name,
    b.title,
    b.rating,
    b.price_gbp,
    b.price_inr
FROM books AS b
JOIN categories AS c
    ON b.category_id = c.category_id
WHERE b.rating = (
    SELECT MAX(b2.rating)
    FROM books AS b2
    WHERE b2.category_id = b.category_id
)
ORDER BY c.category_name, b.title;
"""

result5 = pd.read_sql(query5, conn)

display(result5)

,category_name,title,rating,price_gbp,price_inr
0,Default,"A Piece of Sky, a Grain of Rice: A Memoir in F...",5,56.76,5988.18
1,Default,The Inefficiency Assassin: Time Management Tac...,5,20.59,2172.24
2,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92
3,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54
5,Young Fiction,Frostbite (Vampire Academy #2),5,29.99,3163.94
6,Young Fiction,Library of Souls (Miss Peregrineâs Peculiar ...,5,48.56,5123.08
7,Young Fiction,Scarlett Epstein Hates It Here,5,43.55,4594.52
8,Young Fiction,Set Me Free,5,17.46,1842.03
9,Young Fiction,The Darkest Lie,5,35.35,3729.42


In [ ]:
print("Query 1 rows:", len(result1))
print("Query 2 rows:", len(result2))
print("Query 3 rows:", len(result3))
print("Query 4 rows:", len(result4))
print("Query 5 rows:", len(result5))

Query 1 rows: 14
Query 2 rows: 10
Query 3 rows: 3
Query 4 rows: 21
Query 5 rows: 11


In [ ]:
books_df = pd.read_sql(
    """
    SELECT
        book_id,
        title,
        price_gbp,
        price_inr,
        rating,
        in_stock,
        category_id
    FROM books
    """,
    conn
)

categories_df = pd.read_sql(
    """
    SELECT
        category_id,
        category_name
    FROM categories
    """,
    conn
)

print("Books DataFrame:")
display(books_df.head())

print("Categories DataFrame:")
display(categories_df)

Books DataFrame:


,book_id,title,price_gbp,price_inr,rating,in_stock,category_id
0,1,Sharp Objects,47.82,5045.01,4,1,1
1,2,"In a Dark, Dark Wood",19.63,2070.96,1,1,1
2,3,The Past Never Ends,56.50,5960.75,4,1,1
3,4,A Murder in Time,16.64,1755.52,1,1,1
4,5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.55,4,1,1


Categories DataFrame:


,category_id,category_name
0,1,Mystery
1,2,Young Fiction
2,3,Default


In [ ]:
merged_df = pd.merge(
    books_df,
    categories_df,
    on="category_id",
    how="inner"
)

display(merged_df.head())

,book_id,title,price_gbp,price_inr,rating,in_stock,category_id,category_name
0,1,Sharp Objects,47.82,5045.01,4,1,1,Mystery
1,2,"In a Dark, Dark Wood",19.63,2070.96,1,1,1,Mystery
2,3,The Past Never Ends,56.50,5960.75,4,1,1,Mystery
3,4,A Murder in Time,16.64,1755.52,1,1,1,Mystery
4,5,The Murder of Roger Ackroyd (Hercule Poirot #4),44.10,4652.55,4,1,1,Mystery


In [ ]:
max_rating_per_category = (
    merged_df
    .groupby("category_id")["rating"]
    .transform("max")
)

pandas_join_result = merged_df[
    merged_df["rating"] == max_rating_per_category
][
    [
        "category_name",
        "title",
        "rating",
        "price_gbp",
        "price_inr"
    ]
].sort_values(
    ["category_name", "title"]
).reset_index(drop=True)

display(pandas_join_result)

,category_name,title,rating,price_gbp,price_inr
0,Default,"A Piece of Sky, a Grain of Rice: A Memoir in F...",5,56.76,5988.18
1,Default,The Inefficiency Assassin: Time Management Tac...,5,20.59,2172.24
2,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92
3,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54
5,Young Fiction,Frostbite (Vampire Academy #2),5,29.99,3163.94
6,Young Fiction,Library of Souls (Miss Peregrineâs Peculiar ...,5,48.56,5123.08
7,Young Fiction,Scarlett Epstein Hates It Here,5,43.55,4594.52
8,Young Fiction,Set Me Free,5,17.46,1842.03
9,Young Fiction,The Darkest Lie,5,35.35,3729.42


In [ ]:
sql_join_result = result5.copy()

sql_join_result = sql_join_result.sort_values(
    ["category_name", "title"]
).reset_index(drop=True)

display(sql_join_result)

,category_name,title,rating,price_gbp,price_inr
0,Default,"A Piece of Sky, a Grain of Rice: A Memoir in F...",5,56.76,5988.18
1,Default,The Inefficiency Assassin: Time Management Tac...,5,20.59,2172.24
2,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92
3,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54
5,Young Fiction,Frostbite (Vampire Academy #2),5,29.99,3163.94
6,Young Fiction,Library of Souls (Miss Peregrineâs Peculiar ...,5,48.56,5123.08
7,Young Fiction,Scarlett Epstein Hates It Here,5,43.55,4594.52
8,Young Fiction,Set Me Free,5,17.46,1842.03
9,Young Fiction,The Darkest Lie,5,35.35,3729.42


In [ ]:
print("SQL result:")
display(sql_join_result)

print("\nPandas merge result:")
display(pandas_join_result)

print("\nAre both results equivalent?")

print(
    sql_join_result.equals(pandas_join_result)
)

SQL result:


,category_name,title,rating,price_gbp,price_inr
0,Default,"A Piece of Sky, a Grain of Rice: A Memoir in F...",5,56.76,5988.18
1,Default,The Inefficiency Assassin: Time Management Tac...,5,20.59,2172.24
2,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92
3,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54
5,Young Fiction,Frostbite (Vampire Academy #2),5,29.99,3163.94
6,Young Fiction,Library of Souls (Miss Peregrineâs Peculiar ...,5,48.56,5123.08
7,Young Fiction,Scarlett Epstein Hates It Here,5,43.55,4594.52
8,Young Fiction,Set Me Free,5,17.46,1842.03
9,Young Fiction,The Darkest Lie,5,35.35,3729.42



Pandas merge result:


,category_name,title,rating,price_gbp,price_inr
0,Default,"A Piece of Sky, a Grain of Rice: A Memoir in F...",5,56.76,5988.18
1,Default,The Inefficiency Assassin: Time Management Tac...,5,20.59,2172.24
2,Mystery,A Time of Torment (Charlie Parker #14),5,48.35,5100.92
3,Mystery,The Bachelor Girl's Guide to Murder (Herringfo...,5,52.30,5517.65
4,Mystery,What Happened on Beale Street (Secrets of the ...,5,25.37,2676.54
5,Young Fiction,Frostbite (Vampire Academy #2),5,29.99,3163.94
6,Young Fiction,Library of Souls (Miss Peregrineâs Peculiar ...,5,48.56,5123.08
7,Young Fiction,Scarlett Epstein Hates It Here,5,43.55,4594.52
8,Young Fiction,Set Me Free,5,17.46,1842.03
9,Young Fiction,The Darkest Lie,5,35.35,3729.42



Are both results equivalent?
True


In [ ]:
from pathlib import Path

output_dir = Path("outputs")
output_dir.mkdir(exist_ok=True)

queries = {
    "Query 1 - SELECT WHERE BETWEEN ORDER BY": (query1, result1),
    "Query 2 - ORDER BY LIMIT": (query2, result2),
    "Query 3 - DISTINCT JOIN": (query3, result3),
    "Query 4 - IN": (query4, result4),
    "Query 5 - JOIN": (query5, result5)
}

with open(
    output_dir / "query_outputs.txt",
    "w",
    encoding="utf-8"
) as f:

    for name, (query, result) in queries.items():

        f.write("=" * 80 + "\n")
        f.write(name + "\n")
        f.write("=" * 80 + "\n\n")

        f.write("SQL Query:\n")
        f.write(query.strip() + "\n\n")

        f.write("Output:\n")
        f.write(result.to_string(index=False))
        f.write("\n\n")

print("query_outputs.txt created successfully!")

query_outputs.txt created successfully!


In [ ]:
with open(
    output_dir / "pandas_comparison.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write("pd.read_sql() JOIN result\n")
    f.write("=" * 60 + "\n")
    f.write(sql_join_result.to_string(index=False))

    f.write("\n\n")

    f.write("pd.merge() JOIN result\n")
    f.write("=" * 60 + "\n")
    f.write(pandas_join_result.to_string(index=False))

    f.write("\n\n")

    f.write(
        "Equivalent: "
        + str(sql_join_result.equals(pandas_join_result))
    )

print("pandas_comparison.txt created successfully!")

pandas_comparison.txt created successfully!


In [ ]:
df.to_csv(
    output_dir / "cleaned_books.csv",
    index=False
)

print("cleaned_books.csv created!")

cleaned_books.csv created!


In [ ]:
from pathlib import Path
import shutil

project_dir = Path("/content/data_pipeline")
outputs_dir = project_dir / "outputs"

project_dir.mkdir(exist_ok=True)
outputs_dir.mkdir(exist_ok=True)

print("Created:", project_dir)

Created: /content/data_pipeline


In [ ]:
# Copy SQLite database
shutil.copy(
    "/content/books_catalog.db",
    project_dir / "books_catalog.db"
)

# Copy generated outputs
for file in Path("/content/outputs").glob("*"):
    shutil.copy(file, outputs_dir / file.name)

print("Files copied successfully!")

for path in project_dir.rglob("*"):
    if path.is_file():
        print(path.relative_to(project_dir))

Files copied successfully!
books_catalog.db
outputs/cleaned_books.csv
outputs/query_outputs.txt
outputs/pandas_comparison.txt


In [ ]:
readme = """
# Data Pipeline — Book Catalog

## Objective

This module implements a complete data-engineering pipeline:

Scrape → Clean → Convert → Store → Query → Validate

The source is Books to Scrape, a public website designed for scraping
practice.

Source:
https://books.toscrape.com/

## Scraping Scope

The pipeline scrapes one listing page from each of three categories:

- Mystery
- Young Fiction
- Default

Each category contributes 20 books, producing at least 60 books across
3 categories.

The scraping is performed programmatically using:

- requests
- BeautifulSoup

No manual copy-pasting is used.

## Data Cleaning

### Price

The GBP currency symbol is removed and the value is converted to float.

Column:

price_gbp

### Rating

Text ratings are converted as follows:

One = 1
Two = 2
Three = 3
Four = 4
Five = 5

Column:

rating

### Availability

Availability text is converted to boolean:

In stock = True
Out of stock = False

SQLite stores the boolean as:

1 = True
0 = False

Column:

in_stock

### Parsing failures

Numeric parsing failures are handled using median imputation.

For invalid availability values, the affected row is dropped because
median imputation is not meaningful for a boolean field.

The pipeline therefore does not crash because of an individual malformed row.

## Currency Conversion

The required project-defined fixed conversion rate is:

1 GBP = 105.50 INR

This is an artificial fixed baseline for the assignment.

No live exchange-rate API is used.

price_inr is calculated as:

price_inr = price_gbp * 105.50

The result is rounded to two decimal places.

## Database Design

A normalized SQLite database is used.

### categories

- category_id INTEGER PRIMARY KEY
- category_name TEXT UNIQUE NOT NULL

### books

- book_id INTEGER PRIMARY KEY
- title TEXT NOT NULL
- price_gbp REAL NOT NULL
- price_inr REAL NOT NULL
- rating INTEGER NOT NULL
- in_stock INTEGER NOT NULL
- category_id INTEGER FOREIGN KEY

Relationship:

books.category_id → categories.category_id

The category name is stored once in the categories table instead of being
repeated for every book.

## SQL Queries

Five SQL queries are executed.

1. SELECT + WHERE + BETWEEN + ORDER BY
2. ORDER BY + LIMIT
3. DISTINCT + JOIN
4. IN + ORDER BY
5. JOIN to identify highest-rated books per category

The SQL query strings and their outputs are saved in:

outputs/query_outputs.txt

## Pandas Validation

The JOIN query is reproduced in pandas using:

pd.merge()

The SQL result is read using:

pd.read_sql()

Both results are compared using DataFrame.equals().

The comparison output is saved in:

outputs/pandas_comparison.txt

The expected comparison result is:

Equivalent: True

## Generated Files

- books_catalog.db
- outputs/cleaned_books.csv
- outputs/query_outputs.txt
- outputs/pandas_comparison.txt

## Running the Pipeline

Install dependencies:

pip install requests beautifulsoup4 pandas

Then run the notebook or Python scraping script.

The pipeline recreates the SQLite database from the source data.

## Git Requirement

The repository must contain:

- a feature branch
- at least two commits on that feature branch
- a merge back into main

Example:

git checkout -b feature/data-pipeline

git add data_pipeline
git commit -m "feat: add catalog scraping and cleaning"

git add data_pipeline
git commit -m "feat: add SQLite schema and SQL validation"

git checkout main
git merge --no-ff feature/data-pipeline -m "merge: data pipeline module"

Verify with:

git log --oneline --graph --decorate --all
"""

(project_dir / "README.md").write_text(
    readme.strip(),
    encoding="utf-8"
)

print("README.md created successfully!")

README.md created successfully!


In [ ]:
requirements = """requests
beautifulsoup4
pandas
"""

(project_dir / "requirements.txt").write_text(
    requirements,
    encoding="utf-8"
)

print("requirements.txt created!")

requirements.txt created!


In [ ]:
sql_text = f"""
-- Query 1: SELECT / WHERE / BETWEEN / ORDER BY
{query1.strip()}

-- Query 2: ORDER BY / LIMIT
{query2.strip()}

-- Query 3: DISTINCT / JOIN
{query3.strip()}

-- Query 4: IN
{query4.strip()}

-- Query 5: JOIN
{query5.strip()}
"""

(project_dir / "sql_queries.sql").write_text(
    sql_text.strip(),
    encoding="utf-8"
)

print("sql_queries.sql created!")

sql_queries.sql created!


In [ ]:
print("FINAL /data_pipeline STRUCTURE")
print("=" * 50)

for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(project_dir))

FINAL /data_pipeline STRUCTURE
README.md
books_catalog.db
outputs/cleaned_books.csv
outputs/pandas_comparison.txt
outputs/query_outputs.txt
requirements.txt
sql_queries.sql


In [ ]:
from pathlib import Path

print("Files created:")
for file in Path("/content").glob("*"):
    print(file)

Files created:
/content/.config
/content/books_catalog.db
/content/outputs
/content/drive
/content/data_pipeline
/content/sample_data


In [ ]:
from pathlib import Path

project_dir = Path("/content/data_pipeline")

print("DATA_PIPELINE CONTENTS")
print("=" * 50)

for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(project_dir))

DATA_PIPELINE CONTENTS
README.md
books_catalog.db
outputs/cleaned_books.csv
outputs/pandas_comparison.txt
outputs/query_outputs.txt
requirements.txt
sql_queries.sql


In [ ]:
from pathlib import Path

readme = Path("/content/data_pipeline/README.md")

print(readme.read_text(encoding="utf-8"))

# Data Pipeline — Book Catalog

## Objective

This module implements a complete data-engineering pipeline:

Scrape → Clean → Convert → Store → Query → Validate

The source is Books to Scrape, a public website designed for scraping
practice.

Source:
https://books.toscrape.com/

## Scraping Scope

The pipeline scrapes one listing page from each of three categories:

- Mystery
- Young Fiction
- Default

Each category contributes 20 books, producing at least 60 books across
3 categories.

The scraping is performed programmatically using:

- requests
- BeautifulSoup

No manual copy-pasting is used.

## Data Cleaning

### Price

The GBP currency symbol is removed and the value is converted to float.

Column:

price_gbp

### Rating

Text ratings are converted as follows:

One = 1
Two = 2
Three = 3
Four = 4
Five = 5

Column:

rating

### Availability

Availability text is converted to boolean:

In stock = True
Out of stock = False

SQLite stores the boolean as:

1 = True
0 = False

Column:

i

In [ ]:
from pathlib import Path

project_dir = Path("/content/data_pipeline")

print("FINAL DATA_PIPELINE FILES")
print("=" * 50)

for path in sorted(project_dir.rglob("*")):
    if path.is_file():
        print(path.relative_to(project_dir))

FINAL DATA_PIPELINE FILES
README.md
books_catalog.db
outputs/cleaned_books.csv
outputs/pandas_comparison.txt
outputs/query_outputs.txt
requirements.txt
sql_queries.sql


In [ ]:
print("Total books:", len(df))
print("Categories:", df["category"].nunique())
print(df["category"].unique())

Total books: 60
Categories: 3
['Mystery' 'Young Fiction' 'Default']


In [ ]:
import sqlite3

conn = sqlite3.connect("books_catalog.db")

print(pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
))

         name
0  categories
1       books


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

drive.mount('/content/drive')

src = Path('/content/data_pipeline')
dst = Path('/content/drive/MyDrive/zepto-assignment/data_pipeline')

dst.mkdir(parents=True, exist_ok=True)

for item in src.iterdir():
    target = dst / item.name

    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

print("Copied successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Copied successfully!


In [ ]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in /content/data_pipeline:")
if os.path.exists("/content/data_pipeline"):
    for root, dirs, files in os.walk("/content/data_pipeline"):
        print(root)
        for f in files:
            print("   ", f)
else:
    print("data_pipeline folder NOT FOUND")

Current folder:
/content

Files in /content/data_pipeline:
/content/data_pipeline
    books_catalog.db
    README.md
    requirements.txt
    sql_queries.sql
/content/data_pipeline/outputs
    cleaned_books.csv
    query_outputs.txt
    pandas_comparison.txt


In [ ]:
from google.colab import drive
from pathlib import Path
import shutil

# Google Drive connect
drive.mount('/content/drive')

# Source: Colab lo already unna files
src = Path('/content/data_pipeline')

# Destination: nee Google Drive folder
dst = Path('/content/drive/MyDrive/zepto-assignment/data_pipeline')

# Folder create
dst.mkdir(parents=True, exist_ok=True)

# Files + outputs folder copy
for item in src.iterdir():
    target = dst / item.name

    if item.is_dir():
        shutil.copytree(item, target, dirs_exist_ok=True)
    else:
        shutil.copy2(item, target)

print("✅ All data_pipeline files copied to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ All data_pipeline files copied to Google Drive!


In [ ]:
from pathlib import Path

dst = Path('/content/drive/MyDrive/zepto-assignment/data_pipeline')

print("FILES IN GOOGLE DRIVE:")
for path in sorted(dst.rglob("*")):
    print(path.relative_to(dst))

FILES IN GOOGLE DRIVE:
README.md
books_catalog.db
outputs
outputs/cleaned_books.csv
outputs/pandas_comparison.txt
outputs/query_outputs.txt
requirements.txt
sql_queries.sql
